In [1]:
import requests

In [ ]:
filename = "/home/john/streamtest.h5"  # the domain path we will use when querying the server
endpoint = "http://localhost:5101"     # the server address

In [ ]:
# get information about the domain
req = endpoint + "/"
headers = {"X-Hdf-domain": filename}
r = requests.get(req, headers=headers)
if r.status_code == 200:
    print(r.json())
else:
    print(f"Error: {r.status_code} {r.reason}")

{'root': 'g-8ee0a8e7-48db9cd8-0668-206fc0-531450', 'class': 'domain', 'owner': 'john', 'created': 1762872533.3198256, 'limits': {'min_chunk_size': 1048576, 'max_chunk_size': 4194304, 'max_request_size': 104857600}, 'compressors': ['blosclz', 'lz4', 'lz4hc', 'gzip', 'zstd', 'deflate'], 'version': '0.9.4', 'lastModified': 1762877270.7273033, 'hrefs': [{'rel': 'self', 'href': 'http://localhost:5101/'}, {'rel': 'database', 'href': 'http://localhost:5101/datasets'}, {'rel': 'groupbase', 'href': 'http://localhost:5101/groups'}, {'rel': 'typebase', 'href': 'http://localhost:5101/datatypes'}, {'rel': 'root', 'href': 'http://localhost:5101/groups/g-8ee0a8e7-48db9cd8-0668-206fc0-531450'}, {'rel': 'acls', 'href': 'http://localhost:5101/acls'}, {'rel': 'parent', 'href': 'http://localhost:5101/?domain=hsdstest/home/john'}]}


In [ ]:
# get the names and ids of all sensors in the domain
sensor_ids = {}
req = endpoint + "/groups/"
headers = {"X-Hdf-domain": filename}
params = {"h5path": "/sensors", "include_links": 1}
r = requests.get(req, headers=headers, params=params)
if r.status_code == 200:
    rsp = r.json()
    if "hrefs" in rsp:
        del rsp["hrefs"]
    print(rsp)
    if "links" in rsp:
        print(rsp["links"])
        links = rsp["links"]
        for title in links:
            link = links[title]
            sensor_ids[title] = link["id"]
else:
    print(f"Error: {r.status_code} {r.reason}")

print("---")
print(sensor_ids)

{'id': 'g-8ee0a8e7-48db9cd8-59c0-b6c911-9920e5', 'root': 'g-8ee0a8e7-48db9cd8-0668-206fc0-531450', 'created': 1762872533.390994, 'lastModified': 1762873477.488328, 'linkCount': 4, 'attributeCount': 0, 'links': {'xyz': {'id': 'd-8ee0a8e7-48db9cd8-6598-00bb80-cdf55a', 'class': 'H5L_TYPE_HARD', 'created': 1762872775.4959307}, 'abc': {'id': 'd-8ee0a8e7-48db9cd8-8cef-2f176e-d51b8a', 'class': 'H5L_TYPE_HARD', 'created': 1762872895.7510262}, '123': {'id': 'd-8ee0a8e7-48db9cd8-0fcb-0dd969-2ea579', 'class': 'H5L_TYPE_HARD', 'created': 1762873423.847143}, 'x456': {'id': 'd-8ee0a8e7-48db9cd8-b6ca-6b1deb-914b03', 'class': 'H5L_TYPE_HARD', 'created': 1762873477.488328}}, 'domain': '/home/john/streamtest.h5', 'bucket': 'hsdstest'}
{'xyz': {'id': 'd-8ee0a8e7-48db9cd8-6598-00bb80-cdf55a', 'class': 'H5L_TYPE_HARD', 'created': 1762872775.4959307}, 'abc': {'id': 'd-8ee0a8e7-48db9cd8-8cef-2f176e-d51b8a', 'class': 'H5L_TYPE_HARD', 'created': 1762872895.7510262}, '123': {'id': 'd-8ee0a8e7-48db9cd8-0fcb-0dd9

In [ ]:
# get inforamation about a specific sensor, will return shape, and attributes
req = endpoint + "/datasets/" + sensor_ids["xyz"]
headers = {"X-Hdf-domain": filename}
params = {"include_attrs": 1}
r = requests.get(req, headers=headers, params=params)
if r.status_code == 200:
    rsp = r.json()
    if "hrefs" in rsp:
        del rsp["hrefs"]
    print(rsp)
else:
    print(f"Error: {r.status_code} {r.reason}")



{'id': 'd-8ee0a8e7-48db9cd8-6598-00bb80-cdf55a', 'root': 'g-8ee0a8e7-48db9cd8-0668-206fc0-531450', 'shape': {'class': 'H5S_SIMPLE', 'dims': [200], 'maxdims': [0]}, 'type': {'class': 'H5T_FLOAT', 'base': 'H5T_IEEE_F32LE'}, 'creationProperties': {}, 'layout': {'class': 'H5D_CHUNKED', 'dims': [262144]}, 'attributeCount': 5, 'created': 1762872775.405119, 'lastModified': 1762872775.755864, 'attributes': {'description': {'type': {'class': 'H5T_STRING', 'length': 'H5T_VARIABLE', 'charSet': 'H5T_CSET_UTF8', 'strPad': 'H5T_STR_NULLTERM'}, 'shape': {'class': 'H5S_SCALAR'}, 'value': 'simulated signal', 'created': 1762872775.546479}, 'frequency': {'type': {'class': 'H5T_INTEGER', 'base': 'H5T_STD_I64LE'}, 'shape': {'class': 'H5S_SCALAR'}, 'value': 10, 'created': 1762872775.5961542}, 'scale_factor': {'type': {'class': 'H5T_INTEGER', 'base': 'H5T_STD_I64LE'}, 'shape': {'class': 'H5S_SCALAR'}, 'value': 13, 'created': 1762872775.6468763}, 'update_interval': {'type': {'class': 'H5T_INTEGER', 'base': 'H

In [ ]:
# get sensor data values in the rang start to stop
start = 10
stop = 20
req = endpoint + "/datasets/" + sensor_ids["xyz"] + "/value"
headers = {"X-Hdf-domain": filename}

params = {"select": f"[{start}:{stop}]"}
r = requests.get(req, headers=headers, params=params)
if r.status_code == 200:
    rsp = r.json()
    if "hrefs" in rsp:
        del rsp["hrefs"]
    print(rsp)
else:
    print(f"Error: {r.status_code} {r.reason}")

{'status': 200, 'value': [0.3338927924633026, 0.3827461302280426, 0.4394471347332001, 0.49810507893562317, 0.5655251145362854, 0.6438550353050232, 0.7209286093711853, 0.7870301008224487, 0.8348941206932068, 0.8597061038017273]}
